<a href="https://colab.research.google.com/github/rajeshradhakrishnanmvk/ML2025/blob/master/GhostBeneficiaries_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Scenario: Ghost Beneficiary Detection to Avoid Frequent Mustering for Eligible Retirees (FOOD for THOUGHTS)

**Given**

1.   Payment records, beneficiary data, mortality data, and bank validation data are available.
2.   A process like payment reconciliation (Payment Recon) is established for verifyi ng these datasets.
3. There is a concern about ghost beneficiaries leading to inefficiencies and unnecessary frequent mustering for eligible retirees.

**When**

1.   The system cross-verifies the provided datasets to identify discrepancies.
2.   Potential mismatches or inactive beneficiaries (e.g., deceased individuals) are detected through data comparisons (e.g., mortality data vs. active payment records).
3. Bank validation fails or payment inconsistencies are flagged during payment recon.


**Then**


1. Ghost beneficiaries are identified and flagged in the system for further review or removal.
2. Notifications or reports are generated to exclude flagged beneficiaries from mustering requirements.
3. Eligible retirees experience reduced mustering frequency while maintaining accurate payment records.

In [33]:
# prompt: convert to csv data and load data instead of files '### **1. `payment_records.csv`**
# | TransactionID | BeneficiaryID | Amount | PaymentDate  |
# |---------------|---------------|--------|--------------|
# | T001          | B001          | 1500   | 2024-11-01   |
# | T002          | B002          | 2000   | 2024-11-02   |
# | T003          | B003          | 1800   | 2024-11-03   |
# | T004          | B004          | 1500   | 2024-11-04   |
# | T005          | B005          | 1700   | 2024-11-05   |
# ---
# ### **2. `beneficiary_data.csv`**
# | BeneficiaryID | Name       | VerificationStatus | ActiveStatus | LastVerificationDate |
# |---------------|------------|--------------------|--------------|-----------------------|
# | B001          | Alice      | Verified           | Active       | 2024-10-01           |
# | B002          | Bob        | Verified           | Active       | 2024-10-02           |
# | B003          | Charlie    | Unverified         | Inactive     | 2024-09-15           |
# | B004          |

# prompt: convert to csv data and load data instead of files ### **3. `mortality_data.csv`**
# | BeneficiaryID | DateOfDeath  |
# |---------------|--------------|
# | B003          | 2024-10-10   |
# | B005          | 2024-10-25   |
# ---
# ### **4. `bank_validation_data.csv`**
# | BeneficiaryID | AccountStatus | LastTransactionDate |
# |---------------|---------------|---------------------|
# | B001          | Active        | 2024-11-01         |
# | B002          | Active        | 2024-11-01         |
# | B003          | Inactive      | 2024-09-30         |
# | B004          | Active        | 2024-11-01         |
# | B005          | Inactive      | NULL

import pandas as pd
from io import StringIO

# Create the payment_records DataFrame
payment_records_data = """TransactionID,BeneficiaryID,Amount,PaymentDate
T001,B001,1500,2024-11-01
T002,B002,2000,2024-11-02
T003,B003,1800,2024-11-03
T004,B004,1500,2024-11-04
T005,B005,1700,2024-11-05"""
payment_records = pd.read_csv(StringIO(payment_records_data))
print(payment_records)

# Create the beneficiary_data DataFrame
beneficiary_data_data = """BeneficiaryID,Name,VerificationStatus,ActiveStatus,LastVerificationDate
B001,Alice,Verified,Active,2024-10-01
B002,Bob,Verified,Active,2024-10-02
B003,Charlie,Unverified,Inactive,2024-09-15
B004,NULL,NULL,NULL,NULL""" # Removed the extra comma at the end of this line
beneficiary_data = pd.read_csv(StringIO(beneficiary_data_data))
print(beneficiary_data)

# Create mortality_data.csv in-memory
mortality_data = """BeneficiaryID,DateOfDeath
B003,2024-10-10
B005,2024-10-25"""
mortality_df = pd.read_csv(StringIO(mortality_data))
print("Mortality Data:")
print(mortality_df)

# Create bank_validation_data.csv in-memory
bank_validation_data = """BeneficiaryID,AccountStatus,LastTransactionDate
B001,Active,2024-11-01
B002,Active,2024-11-01
B003,Inactive,2024-09-30
B004,Active,2024-11-01
B005,Inactive,NULL"""
bank_validation_df = pd.read_csv(StringIO(bank_validation_data))
print("\nBank Validation Data:")
print(bank_validation_df)


  TransactionID BeneficiaryID  Amount PaymentDate
0          T001          B001    1500  2024-11-01
1          T002          B002    2000  2024-11-02
2          T003          B003    1800  2024-11-03
3          T004          B004    1500  2024-11-04
4          T005          B005    1700  2024-11-05
  BeneficiaryID     Name VerificationStatus ActiveStatus LastVerificationDate
0          B001    Alice           Verified       Active           2024-10-01
1          B002      Bob           Verified       Active           2024-10-02
2          B003  Charlie         Unverified     Inactive           2024-09-15
3          B004      NaN                NaN          NaN                  NaN
Mortality Data:
  BeneficiaryID DateOfDeath
0          B003  2024-10-10
1          B005  2024-10-25

Bank Validation Data:
  BeneficiaryID AccountStatus LastTransactionDate
0          B001        Active          2024-11-01
1          B002        Active          2024-11-01
2          B003      Inactive        

In [27]:
import pandas as pd
import numpy as np

# Load data
# beneficiaries = pd.read_csv("beneficiary_data.csv")
# payments = pd.read_csv("payment_records.csv")
# mortality = pd.read_csv("mortality_data.csv")
# bank_data = pd.read_csv("bank_validation_data.csv")
# Fix: Use beneficiary_data instead of bank_validation_df
beneficiaries = beneficiary_data
payments = payment_records
mortality = mortality_df
bank_data = bank_validation_df

# Merge datasets
merged = payments.merge(beneficiaries, on="BeneficiaryID", how="left")
merged = merged.merge(mortality, on="BeneficiaryID", how="left")
merged = merged.merge(bank_data, on="BeneficiaryID", how="left")
# Flag irregularities
merged['Issue'] = None
merged.loc[merged['VerificationStatus'] != 'Verified', 'Issue'] = 'Unverified Beneficiary'
merged.loc[merged['ActiveStatus'] == 'Inactive', 'Issue'] = 'Inactive Beneficiary'
merged.loc[merged['PaymentDate'] > merged['DateOfDeath'], 'Issue'] = 'Payment to Deceased'
# Update 'Issue' column for 'No Bank Transaction Found'
merged['Issue'] = np.where(
    (merged['LastTransactionDate'].isnull()) & (merged['Issue'] != 'No Bank Transaction Found'),  # Condition: Update if LastTransactionDate is null and Issue is not already 'No Bank Transaction Found'
    np.where(merged['Issue'].isnull(), 'No Bank Transaction Found', merged['Issue'] + ', No Bank Transaction Found'),  # New value: If Issue is null, set it to 'No Bank Transaction Found'; otherwise, append it to the existing issue
    merged['Issue']  # Existing value if condition is False
)
duplicates = merged[merged.duplicated(['BeneficiaryID', 'PaymentDate'], keep=False)]
duplicates['Issue'] = 'Duplicate Payment'

# Save flagged records
flagged = merged[merged['Issue'].notnull()]
#flagged.to_csv("reconciliation_report.csv", index=False)
print(flagged[['TransactionID', 'BeneficiaryID', 'Issue']])

  TransactionID BeneficiaryID                                           Issue
2          T003          B003                             Payment to Deceased
3          T004          B004                          Unverified Beneficiary
4          T005          B005  Payment to Deceased, No Bank Transaction Found


In [59]:
from transformers import pipeline
from sklearn.metrics.pairwise import cosine_similarity

beneficiaries = beneficiary_data
payments = payment_records
mortality = mortality_df
bank = bank_validation_df

# Merge datasets
merged_1 = payment_records.merge(beneficiaries, on='BeneficiaryID', how='left')
merged_1 = merged_1.merge(mortality, on='BeneficiaryID', how='left')
merged_1 = merged_1.merge(bank, on='BeneficiaryID', how='left')

# Initialize GPT pipeline
gpt_pipe = pipeline('text-classification', model='distilbert-base-uncased')

# Define a function to analyze text data
def analyze_text(text):
    #print("Analyzing text:", text)
    outputs = gpt_pipe(text)
    print("GPT Outputs:", outputs)
    return outputs[0]['label']

# Define a function to detect ghost beneficiaries
def detect_ghost_beneficiaries(merged):
    # Select columns for analysis
    text_columns = ['BeneficiaryID', 'VerificationStatus', 'ActiveStatus', 'LastVerificationDate']
    gpt_texts = []

    # Analyze each beneficiary
    for index, row in merged.iterrows():
        beneficiary_text = ''
        for column in text_columns:
            if not pd.isna(row[column]):
                beneficiary_text += f'{column}: {row[column]}\n'
        gpt_texts.append(beneficiary_text)

    # Use GPT to analyze texts
    results = [analyze_text(text) for text in gpt_texts]
    # print("GPT Results:", results)
    # Flag ghost beneficiaries
    ghost_beneficiaries = [index for index, result in enumerate(results) if result == 'LABEL_1']
    merged.loc[ghost_beneficiaries, 'Issue'] = 'Ghost Beneficiary'

    return merged

# Detect ghost beneficiaries
merged_1 = detect_ghost_beneficiaries(merged_1)
# Save flagged records
flagged = merged_1[merged_1['Issue'].notnull()]
#flagged.to_csv('reconciliation_report.csv', index=False)
print(flagged[['TransactionID', 'BeneficiaryID', 'Issue']])

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


GPT Outputs: [{'label': 'LABEL_0', 'score': 0.5039572715759277}]
GPT Outputs: [{'label': 'LABEL_0', 'score': 0.5040667057037354}]
GPT Outputs: [{'label': 'LABEL_0', 'score': 0.5046039819717407}]
GPT Outputs: [{'label': 'LABEL_0', 'score': 0.5051364898681641}]
GPT Outputs: [{'label': 'LABEL_0', 'score': 0.5065547823905945}]
Empty DataFrame
Columns: [TransactionID, BeneficiaryID, Issue]
Index: []


The issue you're experiencing is because of the GPT model not being trained specifically for the task you're trying to accomplish. The GPT model was trained on a large corpus of text and then fine-tuned on a specific task, like language-to-text generation. Because of this, it may not work well for a specific task like detecting ghost beneficiaries.

To get rid of the "Ghost Beneficiary" issue, you could try a few different things:

1. **Train the GPT model on your own dataset**: Since you're working with a specific domain (beneficiary detection), you could train your own GPT model on a dataset of beneficiary information. This would allow the model to learn specific patterns and features that are relevant to your task.

2. **Use a different model**: There are many different NLP models available, and some may be better suited to your task than GPT. You could try using a different model like BERT, RoBERTa, or even a simple machine learning model like a decision tree or random forest.

3. **Preprocess your data differently**: The way you preprocess your data can greatly impact the performance of your NLP model. You could try different preprocessing techniques, such as tokenization, stemming, or lemmatization, to see if it improves the performance of your model.

Here's an example of how you could train a GPT model on your own dataset:

In [ ]:
from transformers import pipeline

# Load the dataset
train_data = pd.read_csv('train_data.csv')
# "Text","Label"
# "Beneficiary Name: John, Verification Status: Verified, Active Status: Active","0"
# "Beneficiary Name: Jane, Verification Status: Unverified, Active Status: Inactive","1"
# "Beneficiary Name: David, Verification Status: Verified, Active Status: Active","0"
# "Beneficiary Name: Emma, Verification Status: Unverified, Active Status: Active","1"
# "Beneficiary Name: Michael, Verification Status: Verified, Active Status: Inactive","1"
# "Beneficiary Name: Sarah, Verification Status: Verified, Active Status: Active","0"
# "Beneficiary Name: Frank, Verification Status: Unverified, Active Status: Inactive","1"
# "Beneficiary Name: Rachel, Verification Status: Verified, Active Status: Active","0"
# "Beneficiary Name: Mark, Verification Status: Unverified, Active Status: Active","1"

# Create a GPT pipeline
gpt_pipe = pipeline('text-classification', model='distilbert-base-uncased')

# Train the GPT model
for index, row in train_data.iterrows():
    gpt_pipe.train_dataset.append({
        'text': row['text'],
        'label': row['label']
    })

# Use the trained GPT model
trained_gpt_pipe = pipeline('text-classification', model=gpt_pipe.model)

# Use the trained model for predictions
predictions = trained_gpt_pipe(text='some_text')